# Robust Susceptibility Validation Input Generator

`validation_hist.ipynb` と同じ validation 用の摂動点を、HFSS 実行前の `parametric_input.csv` として作成する notebook です。

## 使い方

1. `x_center = []` に 13 次元の中心パラメータを入力します。
2. `N_VALIDATION = 100`, custom perturbation std (`PERTURBATION_SIGMA`), `RANDOM_SEED = 101` の条件で乱数を生成します。従来の `PERTURBATION_STD_RATIO = 0.01` はコメントとして残しています。
3. index `0` を中心点、index `1`〜`100` を摂動点とする `101 x 13` の DataFrame を作ります。
4. `_config.toml` の `[hfss]` / `param_groups` から取得した parameter names を columns にします。
5. timestamp 付き run directory を作成し、その中に `parametric_input.csv` を保存します。


In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
import lib_gp

%load_ext autoreload
%autoreload 2


In [ ]:
# Load parameter metadata from _config.toml.
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)

backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
cfg = app_config.hfss
ROUND_DECIMALS = app_config.runtime.round_decimals

LOWER_BOUNDS = np.asarray(cfg.lower_bounds, dtype=float)
UPPER_BOUNDS = np.asarray(cfg.upper_bounds, dtype=float)
BOUNDS = np.vstack([LOWER_BOUNDS, UPPER_BOUNDS]).T
PARAM_NAMES = list(cfg.param_names)

print("n_params:", cfg.n_params)
print("PARAM_NAMES:", PARAM_NAMES)
print("ROUND_DECIMALS:", ROUND_DECIMALS)


In [ ]:
# ===== User input =====
# cfg.n_params (= 13) と同じ長さの中心パラメータをここに入力してください。
# Example:
# x_center = np.asarray([
#     6.225785,
#     4.065235,
#     9.188137,
#     9.788713,
#     1.000000,
#     1.071733,
#     2.000000,
#     1.456739,
#     1.835538,
#     0.537034,
#     2.000000,
#     3.711260,
#     6.000000,
# ], dtype=float)
x_center = []

N_VALIDATION = 100
# PERTURBATION_STD_RATIO = 0.01  # legacy ratio-based std; kept for reference.
RANDOM_SEED = 101

# Custom per-parameter perturbation standard deviations.
# PARAM_NAMES is loaded from _config.toml in the previous cell, so this dictionary
# is converted to a diagonal covariance matrix in the same parameter order.
CUSTOM_PERTURBATION_STD = {
    "h1": 0.1,
    "h2": 0.1,
    "h3": 0.1,
    "h4": 0.1,
    "h5": 0.1,
    "s1": 0.01,
    "s2": 0.02,
    "s3": 0.02,
    "s4": 0.02,
    "s5": 0.02,
    "a": 0.1,
    "b": 0.1,
    "k": 0.1,
}
CUSTOM_PERTURBATION_STD_ARRAY = np.asarray([CUSTOM_PERTURBATION_STD[name] for name in PARAM_NAMES], dtype=float)
PERTURBATION_SIGMA = np.diag(CUSTOM_PERTURBATION_STD_ARRAY ** 2)


In [ ]:
def validate_x_center(x, n_params, lower_bounds, upper_bounds):
    """Validate and round the user-provided center vector."""
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size != n_params:
        raise ValueError(
            f"x_center must have length cfg.n_params={n_params}, but got length {x.size}. "
            "Set x_center before generating parametric_input.csv."
        )

    below = x < lower_bounds
    above = x > upper_bounds
    if np.any(below) or np.any(above):
        bad = [
            (PARAM_NAMES[i], float(x[i]), float(lower_bounds[i]), float(upper_bounds[i]))
            for i in np.where(below | above)[0]
        ]
        raise ValueError(f"x_center has out-of-bounds values: {bad}")

    return np.round(x, ROUND_DECIMALS)


def build_validation_sigma(custom_sigma=None):
    """Build validation covariance.

    By default this notebook uses ``PERTURBATION_SIGMA`` from the user-input
    cell. If ``custom_sigma`` is None, the legacy ratio-based formula can still
    be used by defining ``PERTURBATION_STD_RATIO`` in the user-input cell.
    """
    if custom_sigma is not None:
        sigma = np.asarray(custom_sigma, dtype=float)
    else:
        bounds_width = UPPER_BOUNDS - LOWER_BOUNDS
        perturbation_std = PERTURBATION_STD_RATIO * bounds_width
        sigma = np.diag(perturbation_std ** 2)

    expected_shape = (cfg.n_params, cfg.n_params)
    if sigma.shape != expected_shape:
        raise ValueError(f"Sigma must have shape {expected_shape}, got {sigma.shape}.")
    return sigma


In [ ]:
x_center = validate_x_center(x_center, cfg.n_params, LOWER_BOUNDS, UPPER_BOUNDS)
VALIDATION_SIGMA = build_validation_sigma(PERTURBATION_SIGMA)
rng = np.random.default_rng(RANDOM_SEED)

# Generate only perturbation rows here. The center row is prepended as index 0 below.
Z_perturbation = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=VALIDATION_SIGMA,
    n_samples=N_VALIDATION,
    bounds=BOUNDS,
    rng=rng,
)
Z_perturbation = np.round(Z_perturbation, ROUND_DECIMALS)

# index 0: center, index 1..N_VALIDATION: perturbations
Z_validation_with_center = np.vstack([x_center, Z_perturbation])
parametric_input_df = pd.DataFrame(Z_validation_with_center, columns=PARAM_NAMES)
parametric_input_df.index.name = "sample_idx"

print("DataFrame shape:", parametric_input_df.shape)
print("Center row index:", parametric_input_df.index[0])
print("Perturbation index range:", f"{parametric_input_df.index[1]}..{parametric_input_df.index[-1]}")
display(parametric_input_df.head())


In [ ]:
# Create a timestamped run directory using the same Backbone directory convention as the other notebooks.
backbone.mkdir()
run_dir = backbone._get_dir_run()

parametric_input_csv = run_dir / "parametric_input.csv"
parametric_input_df.to_csv(parametric_input_csv, index=False)

print(f"Saved parametric input CSV: {parametric_input_csv}")
print(f"Rows x columns: {parametric_input_df.shape[0]} x {parametric_input_df.shape[1]}")


In [ ]:
# Optional: inspect the saved CSV.
saved_parametric_input_df = pd.read_csv(parametric_input_csv)
display(saved_parametric_input_df.head())


## Run parametric HFSS simulations

The cells below execute a simple parametric search over `parametric_input_df`. No Gaussian-process optimization or susceptibility analysis is performed; every row of the generated DataFrame is evaluated once and saved to `results.h5` in the same `output/repeat_1` format used by `main.ipynb`.


In [ ]:
# Initialize the HDF5 storer in the same timestamped run directory.
# If the previous save cell already created run_dir, initStorer reuses it.
backbone.initStorer(mode="w")
run_dir = backbone._get_dir_run()

model_paths, model_paths_str = backbone._get_path_models()
RESULTS_FILE = str(run_dir / Path(_config["io"]["filename_output"]))
TEMP_FILE = str(run_dir / Path(_config["io"]["filename_temp"]))

print("Run directory:", run_dir)
print("HDF5 file:", run_dir / "results.h5")
print("RESULTS_FILE:", RESULTS_FILE)
print("TEMP_FILE:", TEMP_FILE)


In [ ]:
def round_params(params, decimals=ROUND_DECIMALS):
    return np.round(np.asarray(params, dtype=float).flatten(), decimals)


def round_history_row(row, param_names, decimals=ROUND_DECIMALS):
    rounded = dict(row)
    for name in param_names:
        if name in rounded:
            rounded[name] = float(np.round(rounded[name], decimals))
    if "S11" in rounded and pd.notna(rounded["S11"]):
        rounded["S11"] = float(np.round(rounded["S11"], decimals))
    if "Metric" in rounded and pd.notna(rounded["Metric"]):
        rounded["Metric"] = float(np.round(rounded["Metric"], decimals))
    if "gamma" in rounded and pd.notna(rounded["gamma"]):
        rounded["gamma"] = float(np.round(rounded["gamma"], decimals))
    return rounded


def getResult(input_params, param_names, temp_hfss_path, result_file_path):
    """Read one HFSS temp result and append it to the run-level results CSV."""
    df_temp = pd.read_csv(temp_hfss_path)
    header_flag = not os.path.exists(result_file_path)

    try:
        s11_value = df_temp.iloc[-1, -1]
        result_row = dict(zip(param_names, input_params))
        result_row["S11"] = s11_value

        df_result = pd.DataFrame([result_row])
        df_result.to_csv(result_file_path, mode="a", header=header_flag, index=False)

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return True

    except Exception as e:
        print(f"[Error][getResult] Failed to process result: {e}")
        return False


def target_hfss(_config_temp, sim_id, param_names, params):
    """Evaluate one parameter vector through the existing HFSS watcher workflow."""
    params = round_params(params)
    backbone.call_subroutine(
        _config_temp,
        sim_id,
        param_names,
        params,
        value_fmt=f"{{:.{ROUND_DECIMALS}f}}",
    )
    getResult(params, param_names, TEMP_FILE, RESULTS_FILE)
    df_result = pd.read_csv(RESULTS_FILE)
    return float(df_result.iloc[-1]["S11"])


def costFunctionWrapper(param_names, params):
    params = round_params(params, decimals=ROUND_DECIMALS)
    sim_id = backbone._getSimulationID()
    y = target_hfss(_config_temp, sim_id, param_names, params)

    row = dict(zip(param_names, params))
    row["S11"] = float(np.round(y, ROUND_DECIMALS))
    row["Metric"] = np.nan
    row["gamma"] = np.nan
    row["routine_idx"] = 0
    return y, round_history_row(row, param_names)


In [ ]:
# HFSS watcher run-specific config, matching main.ipynb's hand-off format.
_config_temp = {
    "n_simulation": int(len(parametric_input_df)),
    "n_repeats": 1,
    "WATCH_DIR": str(run_dir),
    "INPUT_FILE": str(run_dir / Path(_config["io"]["filename_input"])),
    "MODEL_FILE": model_paths_str,
    "RESULTS_FILE": RESULTS_FILE,
    "TEMP_FILE": TEMP_FILE,
    "DONE_FLAG_FILE": str(run_dir / Path("hfss.done")),
}

done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)

with open(base_dir / Path("_config_HFSS.json"), "w") as f:
    json.dump(_config_temp, f, indent=4)

print(f"Temporarily updated '{base_dir / Path('_config_HFSS.json')}' with run-specific WATCH_DIR for HFSS.")
print("Number of parametric simulations:", len(parametric_input_df))


In [ ]:
# Simple parametric search: evaluate every row in parametric_input_df once.
# No GP optimization and no susceptibility analysis are performed in this loop.
parametric_history = []
start = time.perf_counter()

try:
    for sample_idx, (_, row) in enumerate(parametric_input_df.iterrows(), start=1):
        params = row[PARAM_NAMES].to_numpy(dtype=float)
        print(f"[parametric] HFSS evaluation {sample_idx}/{len(parametric_input_df)}")
        _, result_row = costFunctionWrapper(PARAM_NAMES, params)
        parametric_history.append(result_row)

    df_final = pd.DataFrame(parametric_history)
    df_final[PARAM_NAMES] = df_final[PARAM_NAMES].round(ROUND_DECIMALS)
    df_final["S11"] = df_final["S11"].round(ROUND_DECIMALS)

    df_output = backbone._genOutputDataFrame(df_final)
    if "best" in df_output:
        df_output["best"] = df_output["best"].round(ROUND_DECIMALS)

    parametric_results_csv = run_dir / "parametric_results.csv"
    df_output.to_csv(parametric_results_csv, index=False)
    backbone._addNewDatasetToHDF(df_output.select_dtypes(include=[np.number]), "output", "repeat_1")

    elapsed = time.perf_counter() - start
    best_idx = df_output["S11"].idxmin()
    print("-" * 75)
    print(f"Parametric search finished in {elapsed:.3f} seconds")
    print(f"Best S11: {df_output.loc[best_idx, 'S11']:.10f}")
    print("Best row:")
    display(df_output.loc[[best_idx]])
    print(f"Saved results CSV: {parametric_results_csv}")
    print(f"Saved HDF5: {run_dir / 'results.h5'}")

finally:
    done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
    done_flag_path.touch()

    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True)
    if backbone.h5f:
        backbone.h5f.close()
